In [ ]:
import os
import psycopg
import snowflake.connector as sc
import pandas
import json

In [ ]:
conn_params = {
    'account': "HOFEOUE-CPB80774",
    'user': "GARRETT@ALIGNABLE.COM",
    'authenticator': "externalbrowser",
    'role': "DEVELOPMENT_ROLE",
    'warehouse': "DEV_SMALL",
    'database': "ALIGNABLE_REPORTING",
    'schema':"EVENT_STREAMS",
    'paramstyle': 'qmark'
}

alignable_reporting_conn = sc.connect(**conn_params)
reporting_cursor = alignable_reporting_conn.cursor()

In [ ]:
def query(query_str):
	return reporting_cursor.execute(query_str).fetch_pandas_all()

In [ ]:
test = query("""
	select * from ALIGNABLE_REPORTING.BUSINESSES.BUSINESSES where id = 8;
""")
test["NAME"]

In [ ]:
messages_df = query(f"""
    select m.*, a.variation as test_variant
    from EVENTS.LLM_MESSAGES.LLM_GENERATED_VS_SENT_MESSAGE_EVENTS m
    inner join ALIGNABLE_REPORTING_V2.SPLIT_TESTS.FCT_SPLIT_TEST_ASSIGNMENT a
        on a.business_id = m.source_business_id
    where m.GENERATED_AT >= '2026-04-30'
      and a.split_test_id = 16668
      and a.variation in ('test', 'control')
      and m.GENERATED_AT >= a.assigned_at
      and m.SOURCE_BUSINESS_ID != 17589711
    order by RANDOM()
    limit 1000;
""").reset_index(drop=True)
messages_df.shape

In [ ]:
messages_df.head(1)

In [ ]:
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from anthropic import AnthropicBedrock
from pydantic import BaseModel, Field
from helpers import HAIKU_PROFILE

bedrock_client = AnthropicBedrock(aws_region="us-east-1")

class LeakageClassification(BaseModel):
    contains_leakage: bool = Field(
        description="True if the GENERATED_MESSAGE shows signs that the LLM's prompt, instructions, internal reasoning, or partial/incomplete thoughts leaked into what should be a clean message from one business owner to another. False if the message reads as something a real sender could have written intentionally."
    )
    leakage_type: str = Field(
        description="One of: 'none', 'prompt_leak' (fragments of the instructions/prompt appear in the message, e.g. brackets, field names, 'as the sender'), 'thinking_leak' (model narrating its reasoning, e.g. 'Let me think...', 'Based on the context...', 'I'll craft a message that...'), 'partial_thought' (sentence trails off, ends mid-word, contains placeholder-like text, or reads like unfinished notes), 'meta_commentary' (e.g. 'Here is the message:', 'Sure, I can help', 'Of course!'), 'other' (some other LLM artifact)."
    )
    evidence: str = Field(
        description="A short verbatim quote (≤30 words) from the GENERATED_MESSAGE that demonstrates the leakage. Empty string if contains_leakage is False."
    )

classify_tool = {
    "name": "classify_message",
    "description": "Classify whether a generated message contains LLM leakage artifacts.",
    "input_schema": LeakageClassification.model_json_schema(),
}

CLASSIFY_PROMPT_PREFIX = """\
You review messages auto-generated by an LLM on behalf of small business owners on a professional networking site. The LLM is supposed to produce a clean, polished message from the sender to the target — as if the sender wrote it themselves.

Sometimes the LLM fails and leaks artifacts of its own process into the message. Examples of leakage:
- Fragments of the prompt or instructions appearing in the message (field names, brackets, role descriptions like "as the sender")
- The model narrating its reasoning ("Let me think about how to approach this...", "Based on the context provided...", "I'll craft a friendly message that...")
- Partial thoughts that trail off, end mid-word, or read like unfinished notes-to-self
- Meta-commentary ("Here is the message:", "Sure, I can help draft this", "Of course!")
- Placeholder text that was never substituted (e.g. "{first_name}", "[business name]")

Your job: decide whether the GENERATED_MESSAGE contains any such artifact. A real business owner's message would not.

Be strict about what counts as leakage, but do not flag merely awkward phrasing, generic greetings, or short messages — only flag content that betrays the LLM's internal process or prompt.
"""

def classify_with_retry(message: str, max_retries: int = 8) -> LeakageClassification | None:
    delay = 1.0
    for attempt in range(max_retries):
        try:
            resp = bedrock_client.messages.create(
                model=HAIKU_PROFILE,
                max_tokens=500,
                tools=[classify_tool],
                tool_choice={"type": "tool", "name": "classify_message"},
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": CLASSIFY_PROMPT_PREFIX, "cache_control": {"type": "ephemeral"}},
                        {"type": "text", "text": f"<generated_message>\n{message}\n</generated_message>"},
                    ],
                }],
            )
            tool_use = next(b for b in resp.content if b.type == "tool_use")
            return LeakageClassification.model_validate(tool_use.input)
        except Exception as e:
            err = f"{type(e).__name__}: {e}".lower()
            is_rate_limit = (
                "429" in err
                or "throttl" in err
                or "toomanyrequests" in err
                or "rate" in err
                or "service unavailable" in err
            )
            if attempt == max_retries - 1:
                print(f"  failed after {max_retries} attempts: {type(e).__name__}: {e}")
                return None
            sleep_for = delay + random.uniform(0, delay)
            if not is_rate_limit:
                sleep_for = min(sleep_for, 4.0)
            time.sleep(sleep_for)
            delay = min(delay * 2, 60)
    return None

n = len(messages_df)
results: list[LeakageClassification | None] = [None] * n
print(f"Classifying {n} messages with {HAIKU_PROFILE} on 128 threads...")

with ThreadPoolExecutor(max_workers=128) as ex:
    futures = {
        ex.submit(classify_with_retry, msg): i
        for i, msg in enumerate(messages_df["GENERATED_MESSAGE"])
    }
    for done, fut in enumerate(as_completed(futures), 1):
        results[futures[fut]] = fut.result()
        if done % 25 == 0 or done == n:
            print(f"  [{done}/{n}] complete", end="\r")

print()

leakage_df = messages_df.copy()
leakage_df["contains_leakage"] = [r.contains_leakage if r else None for r in results]
leakage_df["leakage_type"] = [r.leakage_type if r else None for r in results]
leakage_df["leakage_evidence"] = [r.evidence if r else None for r in results]

failures = sum(1 for r in results if r is None)
print(f"Failures: {failures} / {n}")
print()
print("Leakage rate by variant:")
print(leakage_df.groupby("TEST_VARIANT")["contains_leakage"].agg(["mean", "sum", "count"]))
print()
print("Leakage type counts:")
print(leakage_df["leakage_type"].value_counts(dropna=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from math import sqrt

clean_df = leakage_df.dropna(subset=["contains_leakage"]).copy()
clean_df["contains_leakage"] = clean_df["contains_leakage"].astype(bool)
clean_df["SENT_DATE"] = pd.to_datetime(clean_df["SENT_AT"]).dt.date

variants = ["control", "test"]
rate_by_variant = (
    clean_df.groupby("TEST_VARIANT")["contains_leakage"]
    .agg(["mean", "sum", "count"])
    .reindex(variants)
)

leakage_only = clean_df[clean_df["contains_leakage"]]
type_order = ["prompt_leak", "thinking_leak", "partial_thought", "meta_commentary", "other"]
type_by_variant = (
    leakage_only.groupby(["TEST_VARIANT", "leakage_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=variants, columns=type_order, fill_value=0)
)

z = 1.96
def wilson(p, n):
    if n == 0:
        return (0.0, 0.0)
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return (max(0.0, center - half), min(1.0, center + half))

rates_pct = rate_by_variant["mean"].values * 100
counts = rate_by_variant["count"].values
nums = rate_by_variant["sum"].values
ci = np.array([wilson(p, n) for p, n in zip(rate_by_variant["mean"].values, counts)])
err_low = rates_pct - ci[:, 0] * 100
err_high = ci[:, 1] * 100 - rates_pct

bar_colors = {"control": "#9aa5b1", "test": "#3b82f6"}
type_colors = {
    "prompt_leak":      "#ef4444",
    "thinking_leak":    "#f97316",
    "partial_thought":  "#eab308",
    "meta_commentary":  "#a855f7",
    "other":            "#6b7280",
}

fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.45, wspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

# Top-left: leakage rate by variant, with Wilson 95% CI
bars = ax1.bar(
    variants,
    rates_pct,
    yerr=[err_low, err_high],
    capsize=10,
    color=[bar_colors[v] for v in variants],
    edgecolor="black",
    linewidth=0.5,
)
for bar, pct, num, total, hi in zip(bars, rates_pct, nums, counts, err_high):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + hi + 0.05,
        f"{pct:.2f}%\n({int(num)}/{int(total)})",
        ha="center",
        va="bottom",
        fontsize=10,
    )
ax1.set_ylabel("% of messages flagged as containing leakage")
ax1.set_title("LLM-leakage rate by split-test variant\n(Wilson 95% CI)")
ax1.set_ylim(0, max((rates_pct + err_high).max() * 1.6, 0.5))
ax1.grid(axis="y", linestyle="--", alpha=0.4)

# Top-right: stacked counts by leakage type per variant
bottom = np.zeros(len(variants))
for leak_type in type_order:
    vals = type_by_variant[leak_type].values.astype(float)
    if vals.sum() == 0:
        continue
    ax2.bar(
        variants,
        vals,
        bottom=bottom,
        label=leak_type,
        color=type_colors[leak_type],
        edgecolor="white",
        linewidth=0.5,
    )
    for i, v in enumerate(vals):
        if v >= 1:
            ax2.text(
                i,
                bottom[i] + v / 2,
                f"{int(v)}",
                ha="center",
                va="center",
                fontsize=10,
                color="white",
                fontweight="bold",
            )
    bottom += vals

for i, total in enumerate(bottom):
    ax2.text(i, total + 0.1, f"total: {int(total)}", ha="center", va="bottom", fontsize=10)

ax2.set_ylabel("# of flagged messages")
ax2.set_title("Leakage type composition by variant")
ax2.set_ylim(0, max(bottom.max() * 1.3, 2))
ax2.grid(axis="y", linestyle="--", alpha=0.4)
ax2.legend(title="leakage_type", loc="upper left", bbox_to_anchor=(1.02, 1.0))

# Bottom (full width): flagged messages per day by SENT_AT, grouped by variant
dated = clean_df.dropna(subset=["SENT_DATE"]).copy()
dropped_no_sent_at = len(clean_df) - len(dated)

per_day = (
    dated.groupby(["SENT_DATE", "TEST_VARIANT"])
    .agg(flagged=("contains_leakage", "sum"), total=("contains_leakage", "count"))
    .reset_index()
)

if len(per_day) == 0:
    ax3.text(0.5, 0.5, "No SENT_AT values to plot.", ha="center", va="center", transform=ax3.transAxes)
    ax3.set_title("Flagged messages per day (by SENT_AT)")
else:
    all_dates = list(pd.date_range(per_day["SENT_DATE"].min(), per_day["SENT_DATE"].max(), freq="D").date)
    grid = (
        per_day.set_index(["SENT_DATE", "TEST_VARIANT"])
        .reindex(pd.MultiIndex.from_product([all_dates, variants], names=["SENT_DATE", "TEST_VARIANT"]), fill_value=0)
        .reset_index()
    )

    x = np.arange(len(all_dates))
    width = 0.4

    for offset, variant in zip([-width / 2, width / 2], variants):
        sub = grid[grid["TEST_VARIANT"] == variant].sort_values("SENT_DATE")
        flagged_vals = sub["flagged"].values
        total_vals = sub["total"].values
        ax3.bar(
            x + offset,
            flagged_vals,
            width=width,
            color=bar_colors[variant],
            edgecolor="black",
            linewidth=0.4,
            label=f"{variant} (n={int(total_vals.sum())})",
        )
        for xi, fv, tv in zip(x + offset, flagged_vals, total_vals):
            if fv > 0:
                ax3.text(xi, fv + 0.05, f"{int(fv)}/{int(tv)}", ha="center", va="bottom", fontsize=8)

    ax3.set_xticks(x)
    ax3.set_xticklabels([d.strftime("%b %d") for d in all_dates], rotation=45, ha="right")
    ax3.set_ylabel("# of flagged messages sent that day")
    title = "Flagged messages per day by SENT_AT  (label: flagged / total-sampled-that-day)"
    if dropped_no_sent_at:
        title += f"  — {dropped_no_sent_at} rows with null SENT_AT excluded"
    ax3.set_title(title)
    ax3.set_ylim(0, max(grid["flagged"].max() * 1.5, 2))
    ax3.grid(axis="y", linestyle="--", alpha=0.4)
    ax3.legend(loc="upper right")

fig.suptitle(
    f"Are LLM thoughts/prompts leaking into WFM messages?  n={int(counts.sum())} classified",
    fontsize=13,
    y=0.995,
)
plt.show()

print("Flagged messages (evidence):")
flagged = clean_df.loc[
    clean_df["contains_leakage"],
    ["SENT_DATE", "TEST_VARIANT", "leakage_type", "leakage_evidence", "GENERATED_MESSAGE"],
].sort_values("SENT_DATE")
for _, row in flagged.iterrows():
    print(f"\n[{row['SENT_DATE']} / {row['TEST_VARIANT']:>7} / {row['leakage_type']}]")
    print(f"  evidence: {row['leakage_evidence']!r}")
    print(f"  message:  {row['GENERATED_MESSAGE'][:200]!r}")

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

flagged_df = (
    leakage_df.loc[
        leakage_df["contains_leakage"] == True,
        ["SENT_AT", "TEST_VARIANT", "leakage_type", "leakage_evidence", "GENERATED_MESSAGE"],
    ]
    .sort_values(["TEST_VARIANT", "SENT_AT"])
    .reset_index(drop=True)
)

def quote_block(text: str) -> str:
    return "\n".join(f"> {line}" if line else ">" for line in str(text).splitlines())

lines = [f"## {len(flagged_df)} flagged messages", ""]
for i, row in flagged_df.iterrows():
    lines.append(f"### #{i + 1} — `{row['TEST_VARIANT']}` / `{row['leakage_type']}`")
    lines.append("")
    lines.append(f"- **sent_at:** `{row['SENT_AT']}`")
    lines.append(f"- **evidence:** {row['leakage_evidence']}")
    lines.append("")
    lines.append("**Generated message:**")
    lines.append("")
    lines.append(quote_block(row["GENERATED_MESSAGE"]))
    lines.append("")
    lines.append("---")
    lines.append("")

markdown_text = "\n".join(lines)

notebook_dir = Path("notebooks/garrett/see_why_wfm")
if not notebook_dir.exists():
    notebook_dir = Path(".")
md_path = notebook_dir / "flagged_messages.md"
md_path.write_text(markdown_text)
print(f"Wrote {len(flagged_df)} flagged messages to {md_path}")

display(Markdown(markdown_text))

In [ ]:
heres_the_message_df = query("""
    select m.*, a.variation as test_variant
    from EVENTS.LLM_MESSAGES.LLM_GENERATED_VS_SENT_MESSAGE_EVENTS m
    inner join ALIGNABLE_REPORTING_V2.SPLIT_TESTS.FCT_SPLIT_TEST_ASSIGNMENT a
        on a.business_id = m.source_business_id
    where m.GENERATED_AT >= '2026-04-30'
      and a.split_test_id = 16668
      and a.variation in ('test', 'control')
      and m.GENERATED_AT >= a.assigned_at
      and m.SOURCE_BUSINESS_ID != 17589711
      and m.GENERATED_MESSAGE ILIKE '%here''s the message%'
    order by m.GENERATED_AT desc;
""").reset_index(drop=True)

print(f"{len(heres_the_message_df)} matching messages since 2026-04-30")
print()
print("By variant:")
print(heres_the_message_df["TEST_VARIANT"].value_counts())
print()
print("By day:")
print(
    heres_the_message_df.assign(day=pandas.to_datetime(heres_the_message_df["GENERATED_AT"]).dt.date)
    .groupby(["day", "TEST_VARIANT"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
heres_the_message_df

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

def quote_block(text: str) -> str:
    return "\n".join(f"> {line}" if line else ">" for line in str(text).splitlines())

sorted_df = heres_the_message_df.sort_values(["TEST_VARIANT", "GENERATED_AT"]).reset_index(drop=True)

lines = [
    f'## {len(sorted_df)} messages containing "here\'s the message" (since 2026-04-30)',
    "",
]
for i, row in sorted_df.iterrows():
    lines.append(f"### #{i + 1} — `{row['TEST_VARIANT']}`")
    lines.append("")
    lines.append(f"- **generated_at:** `{row['GENERATED_AT']}`")
    lines.append(f"- **sent_at:** `{row['SENT_AT']}`")
    lines.append(f"- **source_business_id:** `{row['SOURCE_BUSINESS_ID']}`")
    lines.append(f"- **message_type:** `{row['MESSAGE_TYPE']}`")
    lines.append("")
    lines.append("**Generated message:**")
    lines.append("")
    lines.append(quote_block(row["GENERATED_MESSAGE"]))
    lines.append("")
    lines.append("---")
    lines.append("")

markdown_text = "\n".join(lines)

notebook_dir = Path("notebooks/garrett/see_why_wfm")
if not notebook_dir.exists():
    notebook_dir = Path(".")
md_path = notebook_dir / "heres_the_message_matches.md"
md_path.write_text(markdown_text)
print(f"Wrote {len(sorted_df)} messages to {md_path}")

display(Markdown(markdown_text))